In [1]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import ccf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.stattools import acf, pacf
from scipy.stats import norm
from statsmodels.tsa.arima.model import ARIMA
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

load_dotenv()

True

Suprime warnigs

In [2]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
import numpy as np
import pandas as pd

# Opção 1: ignorar todos os warnings (global)
warnings.filterwarnings("ignore")

# Opção 2: ignorar apenas categorias específicas (recomendado)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Pandas: desativar aviso de SettingWithCopy (use com cuidado)
pd.options.mode.chained_assignment = None

# NumPy: ignorar avisos numéricos (overflow/underflow etc.)
np.seterr(all="ignore")

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

Realiza imports

In [3]:
# Carregar variáveis de ambiente
load_dotenv()

path_dados_tratados = os.getenv('PATH_DADOS_TRATADOS')

resultado_bruto = pd.read_csv(f"{path_dados_tratados}resultado_bruto.csv", index_col=0, parse_dates=True)
lucro_consolidado = pd.read_csv(f"{path_dados_tratados}lucro_consolidado.csv", index_col=0, parse_dates=True)
despesas_operacionais = pd.read_csv(f"{path_dados_tratados}despesas_operacionais.csv", index_col=0, parse_dates=True)
input_receita = {}
exog_vars_estac_lags = {}
dados_indices = {}
for empresa in resultado_bruto.columns:
    input_receita[empresa] = pd.read_csv(f"{path_dados_tratados}input_receita_{empresa}.csv", index_col=0, parse_dates=True)
    input_receita[empresa].index = pd.to_datetime(input_receita[empresa].index, format='%Y-%m-%d')
    exog_vars_estac_lags[empresa] = pd.read_csv(f"{path_dados_tratados}exog_vars_estac_lags_{empresa}.csv", index_col=0, parse_dates=True)
    exog_vars_estac_lags[empresa].index = pd.to_datetime(exog_vars_estac_lags[empresa].index, format='%Y-%m-%d')
    dados_indices[empresa] = pd.read_csv(f"{path_dados_tratados}dados_indices.csv", index_col=1, parse_dates=True, sep=';')
    dados_indices[empresa].drop(columns=['Unnamed: 0'], inplace=True)

In [4]:
exog_vars_filtered = exog_vars_estac_lags['21490'][['Energia Ind.', 'INCC-Brasil-DI', 'IPAOG-DI', 'Juros EUA', 'PIB Corrente',
                                                    'Vol. Impostos','Vol. PIB Mensal', 'Vol. VA Agro Sazonal', 'Vol. VA Agro Trim. Saz.',
                                                    'Vol. VA Ind. Sazonal', 'Vol. VA Ind. Trim. Saz.', 'Vol. VA Serviços', 'ibov', 'iee', 'igpm']]

In [5]:
#dropar todas as colunas que possuem data no nome
for empresa in resultado_bruto.columns:
    exog_vars_estac_lags[empresa] = exog_vars_estac_lags[empresa].loc[:, ~exog_vars_estac_lags[empresa].columns.str.contains('data', case=False)]
    exog_vars_estac_lags[empresa] = exog_vars_estac_lags[empresa].loc[:, ~exog_vars_estac_lags[empresa].columns.str.contains('Data', case=False)]
    exog_vars_estac_lags[empresa] = exog_vars_estac_lags[empresa].loc[:, ~exog_vars_estac_lags[empresa].columns.str.contains('DATA', case=False)]

In [6]:
len(exog_vars_estac_lags['2437'].columns)

414

In [7]:
for empresa in exog_vars_estac_lags.keys():
    print(empresa)
    exog_vars_estac_lags[empresa] = exog_vars_estac_lags[empresa].loc[:, ~exog_vars_estac_lags[empresa].columns.str.contains('lag', case=False)]

2437
18376
20257
21490


In [8]:
dados_indices[empresa].columns


Index(['ibov', 'iee', 'selic', 'vix', 'igpm', 'ipca', 'incc',
       'IPAOG-DI Ipa - todos os itens - Nro. Índice(1477007)',
       'INCC-Brasil-DI-Todos os itens(1464783)',
       'Indicador de Incerteza da Economia Brasil (IIE-Br)(1428452)',
       'Indicador de Incerteza da Economia Brasil - Componente Mídia (Mídia - IIE-Br)(1463188)',
       'Indicador de Incerteza da Economia Brasil - Componente Expactativas (Expec IIE-Br)(1463189)',
       'Indicador Antecedente de Emprego (IA Emp) com ajuste sazonal(1416205)',
       'Indicador Antecedente de Emprego (IA Emp) com ajuste sazonal(1416205) (Variação Percentual em Relação ao Período Anterior)',
       'PIB a preços de 1995(1428699)', 'PIB a preços correntes(1428698)',
       'Índice de volume mensal  - PIB a preços de mercado (1428676)',
       'Índice de volume mensal pela ótica da produção - Valor adicionado a preços básicos(1428677)',
       'Índice de volume mensal pela ótica da produção - Valor adicionado a preços básicos - Agr

In [9]:
for empresa in resultado_bruto.columns:
    dados_indices[empresa] = dados_indices[empresa][['ipca','igpm', 'Índice da Taxa de Câmbio Real - (Real / Iene (Japão))(1003593)', 'TJLP', 'Índice de volume mensal pela ótica da produção com ajuste sazonal - Valor adicionado - Agropecuária(1428684)', 'Indicador Antecedente de Emprego (IA Emp) com ajuste sazonal(1416205)', 'ibov']]

In [10]:
feats = {}
for empresa in exog_vars_estac_lags.keys():
    print(f"\nEmpresa {empresa}")
    feats[empresa] = cria_biblioteca_sindy(
        X=dados_indices[['ipca','ibov']],
        y=resultado_bruto[empresa][9:],
        poly_degree=1,
        interactions=True,
        rolling_windows=[4, 8],
        drop_constants=True
    )
    print(f"Feats criadas: {feats[empresa].shape[1]}")
    display(feats[empresa].head())


Empresa 2437


NameError: name 'cria_biblioteca_sindy' is not defined

Cria não linearidades

In [11]:
def cria_biblioteca_sindy(
    X,
    y=None,
    poly_degree=1,
    interactions=True,
    rolling_windows=None,
    drop_constants=True,
    trig=False,  # novo: adiciona senos e cossenos das colunas numéricas
    lags_y=1    # novo: número de lags de y a serem adicionados (sem vazamento)
):
    """
    Cria apenas não linearidades a partir de X (e, opcionalmente, estatísticas móveis de y).
    """
    import numpy as np
    import pandas as pd
    from itertools import combinations

    if X is None or X.empty:
        raise ValueError("X não pode ser vazio para criação de não linearidades.")

    X_num = X.select_dtypes(include=[np.number]).copy()
    idx = X_num.index
    if y is not None:
        y = pd.Series(y).copy()
        idx = idx.intersection(y.index)
        X_num = X_num.loc[idx]

    # Base de saída
    outs = [X_num]

    # Lags de y (sem vazamento)
    if y is not None and lags_y and lags_y > 0:
        lag_feats = {}
        for k in range(1, int(lags_y) + 1):
            lag_feats[f"y_lag_{k}"] = y.shift(k).loc[idx]
        outs.append(pd.DataFrame(lag_feats, index=idx))

    # Potências
    if poly_degree is not None and poly_degree >= 2:
        for d in range(2, int(poly_degree) + 1):
            pows = X_num.pow(d)
            pows.columns = [f"{c}__pow{d}" for c in X_num.columns]
            outs.append(pows)

    # Interações
    if interactions:
        inter_cols = {}
        cols = list(X_num.columns)
        for a, b in combinations(cols, 2):
            inter_cols[f"{a}__x__{b}"] = X_num[a] * X_num[b]
        if inter_cols:
            outs.append(pd.DataFrame(inter_cols, index=X_num.index))

    # Seno e cosseno (sobre as colunas originais de X)
    if trig:
        sin_df = np.sin(X_num)
        sin_df.columns = [f"{c}__sin" for c in X_num.columns]
        cos_df = np.cos(X_num)
        cos_df.columns = [f"{c}__cos" for c in X_num.columns]
        outs.extend([sin_df, cos_df])

    # Rolling de y (sem vazamento)
    # if y is not None and rolling_windows:
    #     y_lagged = y.shift(1)
    #     roll_feats = {}
    #     for w in rolling_windows:
    #         rm = y_lagged.rolling(w, min_periods=w).mean().loc[idx]
    #         rs = y_lagged.rolling(w, min_periods=w).std().loc[idx]
    #         roll_feats[f"y_roll_mean_{w}"] = rm
    #         roll_feats[f"y_roll_std_{w}"] = rs
    #     outs.append(pd.DataFrame(roll_feats, index=idx))

    feats = pd.concat([c for c in outs if c is not None and not c.empty], axis=1)

    # Limpeza
    feats = feats.dropna(axis=1, how='all')
    if drop_constants and not feats.empty:
        nun = feats.nunique(dropna=True)
        const_cols = nun[nun <= 1].index.tolist()
        if const_cols:
            feats = feats.drop(columns=const_cols, errors='ignore')

    return feats

Aplica modelo SINDy

In [12]:
def _stlsq_fit(X, y, threshold=1000000, max_iter=10, fit_intercept=True, scale=True, verbose=False):
    """
    STLSQ (Sequential Thresholded Least Squares) para y = X @ beta + intercept.
    - Normaliza X e y em [0, 1] antes do ajuste quando `scale=True`.
    - Aplica threshold absoluto em |coef| no espaço normalizado, refitando até convergir.
    - Converte os coeficientes de volta para a escala original ao final.
    Retorna: dict(modelo) com coef_, intercept_, selected_features_, scaler_, history.
    """
    import numpy as np
    import pandas as pd
    from sklearn.preprocessing import MinMaxScaler

    # Garantir DataFrame/Series e limpar constantes/NaN
    X = pd.DataFrame(X).copy()
    y = pd.Series(y).copy()

    # Remover colunas não numéricas e constantes
    X = X.select_dtypes(include=[np.number])
    nun = X.nunique(dropna=True)
    const_cols = nun[nun <= 1].index.tolist()
    if const_cols and verbose:
        print(f"Descartando colunas constantes: {const_cols}")
    X = X.drop(columns=const_cols, errors='ignore')

    # Alinhamento e dropna
    df_all = pd.concat([y.rename('y'), X], axis=1).dropna(axis=0, how='any')
    if df_all.empty:
        raise ValueError("Sem dados após alinhamento/dropna.")
    y_fit = df_all['y']
    X_fit = df_all.drop(columns=['y'])

    # Escalonamento (normalização para [0, 1])
    scaler = None
    y_scaler = None
    X_work = X_fit.values.astype(float)
    y_work = y_fit.values.astype(float)
    if scale:
        scaler = MinMaxScaler()
        X_work = scaler.fit_transform(X_work)
        y_scaler = MinMaxScaler()
        y_work = y_scaler.fit_transform(y_work.reshape(-1, 1)).ravel()

    # Função STLSQ com intercepto
    def _stlsq(A, b, add_intercept):
        if add_intercept:
            A1 = np.column_stack([np.ones((A.shape[0], 1)), A])
            coef_full, *_ = np.linalg.lstsq(A1, b, rcond=None)
            intercept = float(coef_full[0])
            coef = coef_full[1:]
        else:
            coef, *_ = np.linalg.lstsq(A, b, rcond=None)
            intercept = 0.0
        return coef.astype(float), float(intercept)

    coef_s, intercept_s = _stlsq(X_work, y_work, fit_intercept)

    # Máscara de seleção
    mask = np.ones(X_work.shape[1], dtype=bool)
    history = []

    for it in range(int(max_iter)):
        # Threshold em coeficientes atuais (no espaço normalizado)
        small = np.abs(coef_s) < float(threshold)
        new_mask = mask & (~small)

        # Se não houve mudança, encerra
        if new_mask.sum() == mask.sum():
            if verbose:
                print(f"Iter {it+1}: convergência (sem mudança).")
            break

        mask = new_mask
        if mask.sum() == 0:
            if verbose:
                print(f"Iter {it+1}: todos coeficientes abaixo do threshold, parando.")
            break

        coef_s_sub, intercept_s = _stlsq(X_work[:, mask], y_work, fit_intercept)
        # Reconstruir vetor completo
        coef_s = np.zeros(X_work.shape[1], dtype=float)
        coef_s[mask] = coef_s_sub

        if verbose:
            n_keep = int(mask.sum())
            history.append({'iter': it+1, 'n_features': n_keep})
            print(f"Iter {it+1}: mantendo {n_keep} features.")

    coef_scaled = pd.Series(coef_s, index=X_fit.columns)

    # Converter coeficientes para o espaço ORIGINAL (desfazer normalização)
    if scale:
        x_range = pd.Series(scaler.data_range_, index=X_fit.columns).replace(0.0, 1.0)
        x_min = pd.Series(scaler.data_min_, index=X_fit.columns)
        y_range = float(y_scaler.data_range_[0]) if y_scaler is not None else 1.0
        if y_range == 0.0:
            y_range = 1.0
        y_min = float(y_scaler.data_min_[0]) if y_scaler is not None else 0.0
        beta_orig = coef_scaled * (y_range / x_range)
        intercept_orig = y_min + y_range * intercept_s - float((beta_orig * x_min).sum())
    else:
        beta_orig = coef_scaled
        intercept_orig = intercept_s

    selected = beta_orig[beta_orig != 0.0].sort_values(ascending=False)

    model = {
        'coef_': beta_orig,
        'intercept_': intercept_orig,
        'selected_features_': selected.index.tolist(),
        'scaler_': scaler,
        'y_scaler_': y_scaler,
        'threshold': float(threshold),
        'history': history,
        'fit_intercept': bool(fit_intercept),
        'scaled': bool(scale)
    }
    return model

In [13]:
def sindy_fit(y, X_features, threshold=0.1, max_iter=10, fit_intercept=True, scale=True, verbose=False):
    """
    Ajusta SINDy discreto (regressão esparsa) para y_t ~ Theta_t @ coef + intercept.
    Usa a biblioteca de vetores (X_features) criada previamente.
    Retorna: model_dict, X_aligned, y_aligned
    """
    import pandas as pd

    y = pd.Series(y).copy()
    X = pd.DataFrame(X_features).copy()
    # Alinhar e dropna (contemporâneo: y_t com X_t; evitar vazamento pois X contém apenas lags e transf.)
    df_all = pd.concat([y.rename('y'), X], axis=1).dropna(axis=0, how='any')
    if df_all.empty:
        raise ValueError("Sem dados após alinhamento/dropna em sindy_fit.")

    y_aligned = df_all['y']
    X_aligned = df_all.drop(columns=['y'])
    model = _stlsq_fit(
        X_aligned, y_aligned,
        threshold=threshold, max_iter=max_iter,
        fit_intercept=fit_intercept, scale=scale, verbose=verbose
    )
    return model, X_aligned, y_aligned

In [14]:
def sindy_predict(X, model):
    """
    Predição em lote: y_hat = intercept + X[selected] @ coef[selected]
    """
    import numpy as np
    import pandas as pd

    X = pd.DataFrame(X).copy()
    beta = model['coef_']
    b0 = model['intercept_']
    cols = [c for c in X.columns if c in beta.index and beta.loc[c] != 0.0]
    if not cols:
        return pd.Series(b0, index=X.index)  # modelo degenerado (só intercepto)
    y_hat = b0 + np.dot(X[cols].values, beta.loc[cols].values)
    return pd.Series(y_hat, index=X.index, name='SINDy_hat')


In [15]:
def walk_forward_sindy_one_step(
    serie, X_prepared, tamanho_treino_inicial,
    threshold=1000000, max_iter=10, fit_intercept=True, scale=True, verbose=True
):
    """
    Walk-forward one-step sem vazamento: em cada passo i, ajusta SINDy em dados até i-1 e
    prevê y_i usando X_i (contemporâneo). Retorna: rmse_final, last_model, df_resultados
    """
    import numpy as np
    import pandas as pd
    from sklearn.metrics import mean_squared_error

    # Alinhar y e X e remover NaN
    y = pd.Series(serie).copy()
    X = pd.DataFrame(X_prepared).copy()
    df_all = pd.concat([y.rename('y'), X], axis=1).dropna(axis=0, how='any')
    if df_all.empty:
        raise ValueError("Sem dados após alinhamento/dropna para walk-forward SINDy.")
    y_al = df_all['y']
    X_al = df_all.drop(columns=['y'])

    n = len(y_al)
    if tamanho_treino_inicial < 10:
        tamanho_treino_inicial = max(10, tamanho_treino_inicial)
    if tamanho_treino_inicial >= n:
        raise ValueError("tamanho_treino_inicial deve ser menor que o número de amostras válidas.")

    valores_reais, previsoes = [], []
    last_model = None

    if verbose:
        print(f"Iniciando Walk-Forward SINDy: {n - tamanho_treino_inicial} previsões.")

    for i in range(tamanho_treino_inicial, n):
        y_tr = y_al.iloc[:i]
        X_tr = X_al.iloc[:i]
        X_te = X_al.iloc[i:i+1]
        y_real = y_al.iloc[i]

        # Ajusta SINDy no treino e prevê ponto atual
        last_model, _, _ = sindy_fit(
            y_tr, X_tr,
            threshold=threshold, max_iter=max_iter,
            fit_intercept=fit_intercept, scale=scale, verbose=False
        )
        y_hat_i = float(sindy_predict(X_te, last_model).iloc[0])

        previsoes.append(y_hat_i)
        valores_reais.append(float(y_real))

        if verbose and ((i - tamanho_treino_inicial + 1) % 10 == 0 or i == n - 1):
            print(f"Passo {i - tamanho_treino_inicial + 1}/{n - tamanho_treino_inicial} - feats: {len(last_model['selected_features_'])}")

    rmse_final = float(np.sqrt(mean_squared_error(valores_reais, previsoes)))
    df_resultados = pd.DataFrame(
        {'Real': valores_reais, 'Previsao': previsoes},
        index=y_al.index[-len(valores_reais):]
    )

    if verbose:
        print(f"RMSE Final (one-step, SINDy): {rmse_final:.4f}")

    # Plot opcional
    try:
        import matplotlib.pyplot as plt
        # plt.figure(figsize=(14, 7))
        # plt.plot(y_al, label='Série (alinhada)', color='gray', alpha=0.6)
        # plt.plot(df_resultados['Real'], label='Real (Teste)', color='blue', marker='o', linestyle='None', markersize=5)
        # plt.plot(df_resultados['Previsao'], label='Previsão (SINDy)', color='red', linestyle='--')
        # plt.title('Walk-Forward (One-Step) - SINDy'); plt.xlabel('Data'); plt.ylabel('Valor')
        # plt.legend(); plt.grid(True); plt.show()
    except Exception:
        pass

    return rmse_final, last_model, df_resultados

In [16]:
from itertools import combinations
import time
import math
import numpy as np
import pandas as pd

def sindy_grid_search(
    empresa,
    dados_indices_dict,
    resultado_bruto_df,
    poly_degrees=(1, 2, 3),
    lags_range=(1, 2, 3, 4),
    threshold=0.1,
    tamanho_treino_inicial=12,
    max_iter=50,
    fit_intercept=False,
    scale=True,
    interactions=True,
    trig=True,
    progress_every=25,
    verbose=False
    ):
    """
    Explora combinações de biblioteca SINDy para uma empresa.

    Para cada subconjunto não vazio de colunas numéricas de dados_indices[empresa],
    aplica `cria_biblioteca_sindy` variando `poly_degree` (1..3) e `lags_y` (1..4),
    executa `walk_forward_sindy_one_step` e registra o RMSE obtido.

    Retorna
    -------
    df_resultados : pd.DataFrame
        Linhas com empresa, colunas utilizadas, poly_degree, lags_y, rmse e mensagem de erro (se houver).
    modelos : dict
        Dicionário {(empresa, columns_tuple, poly_degree, lags_y): model_dict} do melhor ajuste salvo.
    previsoes : dict
        Dicionário com os DataFrames de previsões correspondentes às execuções válidas.
    """
    inicio = time.time()
    if empresa not in dados_indices_dict:
        raise KeyError(f"Empresa {empresa} não encontrada em dados_indices.")
    if empresa not in resultado_bruto_df.columns:
        raise KeyError(f"Empresa {empresa} não encontrada em resultado_bruto.")

    serie_target = resultado_bruto_df[empresa].copy()
    base_indices = dados_indices_dict[empresa].select_dtypes(include=[np.number]).copy()
    if base_indices.empty:
        raise ValueError(f"dados_indices[{empresa}] não possui colunas numéricas.")

    colunas = list(base_indices.columns)
    total_subsets = sum(math.comb(len(colunas), r) for r in range(1, len(colunas) + 1))
    total_iter = total_subsets * len(poly_degrees) * len(lags_range)
    if verbose:
        print(f"{empresa}: avaliando {total_iter} combinações (colunas={len(colunas)}).")

    resultados = []
    modelos = {}
    previsoes = {}
    iteracao = 0

    def _gera_subsets(cols):
        for r in range(1, len(cols) + 1):
            for combo in combinations(cols, r):
                yield combo

    for subset_cols in _gera_subsets(colunas):
        X_subset = base_indices[list(subset_cols)].copy()
        for poly_degree in poly_degrees:
            for lags_y in lags_range:
                iteracao += 1
                msg_erro = ""
                model = None
                df_res = None
                rmse = np.nan
                try:
                    biblioteca = cria_biblioteca_sindy(
                        X=X_subset,
                        y=serie_target,
                        poly_degree=poly_degree,
                        interactions=interactions,
                        trig=trig,
                        lags_y=lags_y
                    )
                    if biblioteca.empty:
                        raise ValueError("Biblioteca SINDy vazia após geração de features.")
                    rmse, model, df_res = walk_forward_sindy_one_step(
                        serie=serie_target,
                        X_prepared=biblioteca,
                        tamanho_treino_inicial=tamanho_treino_inicial,
                        threshold=threshold,
                        max_iter=max_iter,
                        fit_intercept=fit_intercept,
                        scale=scale,
                        verbose=False
                    )
                except Exception as exc:
                    msg_erro = str(exc)
                else:
                    chave = (empresa, tuple(subset_cols), poly_degree, lags_y)
                    modelos[chave] = model
                    previsoes[chave] = df_res
                resultados.append({
                    'empresa': empresa,
                    'columns': tuple(subset_cols),
                    'poly_degree': poly_degree,
                    'lags_y': lags_y,
                    'rmse': rmse,
                    'error': msg_erro
                })
                if verbose and progress_every and iteracao % progress_every == 0:
                    tempo_decorrido = time.time() - inicio
                    print(
                        f"{empresa}: {iteracao}/{total_iter} combinações | "
                        f"tempo decorrido: {tempo_decorrido/60:.1f} min"
                    )

    df_resultados = pd.DataFrame(resultados)
    df_resultados = df_resultados.sort_values(by='rmse', ascending=True, na_position='last').reset_index(drop=True)

    if verbose:
        tempo_total = time.time() - inicio
        print(f"{empresa}: busca concluída em {tempo_total/60:.1f} min.")

    return df_resultados, modelos, previsoes

In [17]:
# Configurar busca SINDy (ajuste os valores conforme necessário)
try:
    empresa_grid = '20257'
except NameError:
    empresa_grid = resultado_bruto.columns[0]

poly_range = range(1, 4)   # 1, 2, 3
lags_range = range(1, 5)   # 1, 2, 3, 4
threshold_grid = 0.1
tamanho_treino_grid = 12
max_iter_grid = 50

resultados_sindy_grid, modelos_sindy_grid, previsoes_sindy_grid = sindy_grid_search(
    empresa=empresa_grid,
    dados_indices_dict=dados_indices,
    resultado_bruto_df=resultado_bruto,
    poly_degrees=poly_range,
    lags_range=lags_range,
    threshold=threshold_grid,
    tamanho_treino_inicial=tamanho_treino_grid,
    max_iter=max_iter_grid,
    fit_intercept=False,
    scale=True,
    interactions=True,
    trig=True,
    progress_every=50,
    verbose=True
)

display(resultados_sindy_grid.head(10))
if not resultados_sindy_grid.empty:
    melhor_config = resultados_sindy_grid.iloc[0].copy()
    print("\nMelhor combinação encontrada:")
    print(melhor_config)

20257: avaliando 1524 combinações (colunas=7).
20257: 50/1524 combinações | tempo decorrido: 0.4 min
20257: 50/1524 combinações | tempo decorrido: 0.4 min
20257: 100/1524 combinações | tempo decorrido: 0.9 min
20257: 100/1524 combinações | tempo decorrido: 0.9 min
20257: 150/1524 combinações | tempo decorrido: 1.4 min
20257: 150/1524 combinações | tempo decorrido: 1.4 min
20257: 200/1524 combinações | tempo decorrido: 1.8 min
20257: 200/1524 combinações | tempo decorrido: 1.8 min
20257: 250/1524 combinações | tempo decorrido: 2.3 min
20257: 250/1524 combinações | tempo decorrido: 2.3 min
20257: 300/1524 combinações | tempo decorrido: 2.8 min
20257: 300/1524 combinações | tempo decorrido: 2.8 min
20257: 350/1524 combinações | tempo decorrido: 3.2 min
20257: 350/1524 combinações | tempo decorrido: 3.2 min
20257: 400/1524 combinações | tempo decorrido: 3.8 min
20257: 400/1524 combinações | tempo decorrido: 3.8 min
20257: 450/1524 combinações | tempo decorrido: 4.3 min
20257: 450/1524 comb

,empresa,columns,poly_degree,lags_y,rmse,error
0,20257,"(ibov,)",1,3,1.838623e+08,
1,20257,(Indicador Antecedente de Emprego (IA Emp) com...,1,3,1.883604e+08,
2,20257,"(ibov,)",1,2,1.892038e+08,
3,20257,(Índice da Taxa de Câmbio Real - (Real / Iene ...,1,3,1.941289e+08,
4,20257,"(ibov,)",1,1,1.954073e+08,
5,20257,(Índice da Taxa de Câmbio Real - (Real / Iene ...,2,3,1.982526e+08,
6,20257,(Índice de volume mensal pela ótica da produçã...,1,3,2.017577e+08,
7,20257,(Índice de volume mensal pela ótica da produçã...,1,2,2.023018e+08,
8,20257,(Indicador Antecedente de Emprego (IA Emp) com...,1,2,2.043817e+08,
9,20257,(Índice de volume mensal pela ótica da produçã...,1,1,2.068139e+08,



Melhor combinação encontrada:
empresa                   20257
columns                 (ibov,)
poly_degree                   1
lags_y                        3
rmse           183862277.156243
error                          
Name: 0, dtype: object


In [18]:
resultados_sindy_grid.to_csv(f"{path_dados_tratados}sindy_grid_search_resultados_{empresa_grid}.csv", index=False)

In [19]:
lib_sindy = {}
for empresa in resultado_bruto.columns:
    lib_sindy[empresa] = cria_biblioteca_sindy(
        X=dados_indices[empresa][['ipca','ibov']],
        y=resultado_bruto[empresa],    # mesma defasagem usada antes
        poly_degree=2,
        interactions=True,
        trig=True,
        lags_y=2
    )

In [20]:
#Rodar para cada empresa com a biblioteca já construída (feats[empresa])
sindy_rmse = {}
sindy_model = {}
sindy_preds = {}
for empresa in lib_sindy.keys():
    print(f"\nEmpresa {empresa}")
    rmse, model, df_res = walk_forward_sindy_one_step(
        serie=resultado_bruto[empresa][3:],    # mesma defasagem usada antes
        X_prepared=lib_sindy[empresa][3:],
        tamanho_treino_inicial=12,
        threshold=0.1,        # ajuste o limiar de sparsidade
        max_iter=50,
        fit_intercept=False,
        scale=True,
        verbose=True
    )
    sindy_rmse[empresa] = rmse
    sindy_model[empresa] = model
    sindy_preds[empresa] = df_res


Empresa 2437
Iniciando Walk-Forward SINDy: 44 previsões.
Passo 10/44 - feats: 7
Passo 10/44 - feats: 7
Passo 20/44 - feats: 6
Passo 20/44 - feats: 6
Passo 30/44 - feats: 6
Passo 30/44 - feats: 6
Passo 40/44 - feats: 4
Passo 44/44 - feats: 6
RMSE Final (one-step, SINDy): 8263077942.3661

Empresa 18376
Iniciando Walk-Forward SINDy: 44 previsões.
Passo 40/44 - feats: 4
Passo 44/44 - feats: 6
RMSE Final (one-step, SINDy): 8263077942.3661

Empresa 18376
Iniciando Walk-Forward SINDy: 44 previsões.
Passo 10/44 - feats: 10
Passo 10/44 - feats: 10
Passo 20/44 - feats: 9
Passo 20/44 - feats: 9
Passo 30/44 - feats: 9
Passo 30/44 - feats: 9
Passo 40/44 - feats: 7
Passo 44/44 - feats: 4
RMSE Final (one-step, SINDy): 2175696112.6595

Empresa 20257
Iniciando Walk-Forward SINDy: 44 previsões.
Passo 40/44 - feats: 7
Passo 44/44 - feats: 4
RMSE Final (one-step, SINDy): 2175696112.6595

Empresa 20257
Iniciando Walk-Forward SINDy: 44 previsões.
Passo 10/44 - feats: 9Passo 10/44 - feats: 9
Passo 20/44 - f

In [21]:
sindy_model[empresa]

{'coef_': ipca             3.115879e+13
 ibov             0.000000e+00
 y_lag_1          5.301563e-01
 y_lag_2          0.000000e+00
 ipca__pow2       0.000000e+00
 ibov__pow2       0.000000e+00
 ipca__x__ibov    3.508314e+05
 ipca__sin       -3.118676e+13
 ibov__sin        0.000000e+00
 ipca__cos        0.000000e+00
 ibov__cos        0.000000e+00
 dtype: float64,
 'intercept_': 298278153.07733154,
 'selected_features_': ['ipca', 'ipca__x__ibov', 'y_lag_1', 'ipca__sin'],
 'scaler_': MinMaxScaler(),
 'y_scaler_': MinMaxScaler(),
 'threshold': 0.1,
 'history': [],
 'fit_intercept': False,
 'scaled': True}

In [22]:
# ...existing code...
import os
import pandas as pd

def sindy_coef_dataframe(model, only_selected=True):
    """
    Constrói um DataFrame com os coeficientes do modelo SINDy:
    - feature, coef, abs_coef, selected
    - inclui o intercepto como linha 'INTERCEPT'
    """
    if model is None:
        return pd.DataFrame(columns=['feature','coef','abs_coef','selected'])

    beta = model.get('coef_', pd.Series(dtype=float))
    if isinstance(beta, pd.Series):
        df = beta.rename('coef').to_frame()
    else:
        df = pd.Series(beta, name='coef').to_frame()

    df['abs_coef'] = df['coef'].abs()
    df['selected'] = df['coef'] != 0.0

    if only_selected:
        df = df[df['selected']]

    # Intercepto como linha separada
    intercept = float(model.get('intercept_', 0.0))
    intercept_row = pd.DataFrame(
        {'feature': ['INTERCEPT'], 'coef': [intercept], 'abs_coef': [abs(intercept)], 'selected': [True]}
    ).set_index('feature')

    # Reordenar por |coef|
    df = pd.concat([intercept_row, df], axis=0)
    df = df.sort_values('abs_coef', ascending=False)
    df.index.name = 'feature'
    return df

def sindy_coef_todos(sindy_model, only_selected=True):
    """
    Concatena coeficientes de todas as empresas em um único DataFrame (long).
    Índice: empresa, feature.
    """
    frames = []
    for emp, model in sindy_model.items():
        df_emp = sindy_coef_dataframe(model, only_selected=only_selected).copy()
        df_emp.insert(0, 'empresa', emp)
        frames.append(df_emp.reset_index())
    if not frames:
        return pd.DataFrame(columns=['empresa','feature','coef','abs_coef','selected']).set_index(['empresa','feature'])
    df_all = pd.concat(frames, axis=0, ignore_index=True)
    return df_all.set_index(['empresa','feature']).sort_values(['empresa','abs_coef'], ascending=[True, False])

def mostrar_coef_por_empresa(sindy_model, top_n=20, only_selected=True):
    """
    Mostra os top_n coeficientes (por |coef|) para cada empresa.
    """
    for emp, model in sindy_model.items():
        print(f"\n== {emp} ==")
        df_emp = sindy_coef_dataframe(model, only_selected=only_selected)
        display(df_emp.head(top_n))

# Exemplo de uso:
# Mostrar top 20 coeficientes por empresa
mostrar_coef_por_empresa(sindy_model, top_n=20, only_selected=True)

# Obter todos coeficientes em uma única tabela e exportar
coef_all = sindy_coef_todos(sindy_model, only_selected=True)
display(coef_all.head(50))  # amostra
# Exportar (opcional)
out_csv = os.path.join(path_dados_tratados, "sindy_coeficientes_por_empresa.csv")
coef_all.to_csv(out_csv, encoding="utf-8", index=True)
print(f"Coeficientes exportados em: {out_csv}")
# ...existing code...


== 2437 ==


,coef,abs_coef,selected
feature,,,
ipca__sin,1.523492e+15,1.523492e+15,True
ipca,-1.523457e+15,1.523457e+15,True
ipca__cos,1.856387e+13,1.856387e+13,True
INTERCEPT,-1.855260e+13,1.855260e+13,True
ipca__pow2,1.735826e+13,1.735826e+13,True
ibov,-7.836698e+04,7.836698e+04,True
ibov__pow2,3.561345e-01,3.561345e-01,True



== 18376 ==


,coef,abs_coef,selected
feature,,,
ipca__sin,-8.622654e+13,8.622654e+13,True
ipca,8.616708e+13,8.616708e+13,True
ipca__cos,1.235607e+12,1.235607e+12,True
INTERCEPT,-1.234396e+12,1.234396e+12,True
ipca__x__ibov,5.890984e+05,5.890984e+05,True



== 20257 ==


,coef,abs_coef,selected
feature,,,
ipca__sin,-1.090468e+14,1.090468e+14,True
ipca,1.090331e+14,1.090331e+14,True
ipca__pow2,-4.790286e+11,4.790286e+11,True
ipca__cos,2.901253e+11,2.901253e+11,True
INTERCEPT,-2.892685e+11,2.892685e+11,True
ipca__x__ibov,1.628392e+05,1.628392e+05,True
ibov,-1.462196e+04,1.462196e+04,True
y_lag_1,2.646881e-01,2.646881e-01,True
y_lag_2,1.848388e-01,1.848388e-01,True



== 21490 ==


,coef,abs_coef,selected
feature,,,
ipca__sin,-3.118676e+13,3.118676e+13,True
ipca,3.115879e+13,3.115879e+13,True
INTERCEPT,2.982782e+08,2.982782e+08,True
ipca__x__ibov,3.508314e+05,3.508314e+05,True
y_lag_1,5.301563e-01,5.301563e-01,True


coef      abs_coef  selected
empresa feature                                            
18376   ipca__sin     -8.622654e+13  8.622654e+13      True
        ipca           8.616708e+13  8.616708e+13      True
        ipca__cos      1.235607e+12  1.235607e+12      True
        INTERCEPT     -1.234396e+12  1.234396e+12      True
        ipca__x__ibov  5.890984e+05  5.890984e+05      True
20257   ipca__sin     -1.090468e+14  1.090468e+14      True
        ipca           1.090331e+14  1.090331e+14      True
        ipca__pow2    -4.790286e+11  4.790286e+11      True
        ipca__cos      2.901253e+11  2.901253e+11      True
        INTERCEPT     -2.892685e+11  2.892685e+11      True
        ipca__x__ibov  1.628392e+05  1.628392e+05      True
        ibov          -1.462196e+04  1.462196e+04      True
        y_lag_1        2.646881e-01  2.646881e-01      True
        y_lag_2        1.848388e-01  1.848388e-01      True
        ibov__pow2     9.373818e-02  9.373818e-02      True
21490   ipca__sin     -3.118676e+13  3.118676e+13      True
        ipca           3.115879e+13  3.115879e+13      True
        INTERCEPT      2.982782e+08  2.982782e+08      True
        ipca__x__ibov  3.508314e+05  3.508314e+05      True
        y_lag_1        5.301563e-01  5.301563e-01      True
2437    ipca__sin      1.523492e+15  1.523492e+15      True
        ipca          -1.523457e+15  1.523457e+15      True
        ipca__cos      1.856387e+13  1.856387e+13      True
        INTERCEPT     -1.855260e+13  1.855260e+13      True
        ipca__pow2     1.735826e+13  1.735826e+13      True
        ibov          -7.836698e+04  7.836698e+04      True
        ibov__pow2     3.561345e-01  3.561345e-01      True

Coeficientes exportados em: C:\workspace_mestrado\dados_tratados\sindy_coeficientes_por_empresa.csv


Hammerstein-Wiener

In [23]:
from sysidentpy.model_structure_selection import FROLS
from sysidentpy.basis_function import Polynomial, Fourier
from sysidentpy.general_estimators import NARX
from sklearn.metrics import mean_squared_error
from sysidentpy.parameter_estimation import LeastSquares
from sklearn.linear_model import BayesianRidge, LinearRegression

In [24]:
xlag = [[8] for _ in range(423)]
np.shape(xlag)

scikit_estimator = BayesianRidge()

modelo_hw = NARX(
    basis_function=Polynomial(2),
    ylag=1, # Lags da saída (componente AR)
    xlag=xlag, # Lags da entrada (componente X)
    model_type="NARMAX", # Usamos NARMAX que é o mais geral. Para H-W puro, o ideal é não ter termos cruzados, mas o algoritmo de seleção (FROLS) cuidará disso.
    base_estimator=scikit_estimator
)

#transforme o dataframe exog_vars_estac_lags['2437'] em uma matriz numpy
X = exog_vars_estac_lags['21490'].iloc[:,1:]
X = X.replace([np.inf, -np.inf], np.nan)  # converte inf para NaN
X = X.fillna(0.0)  # ou use .fillna(X_df.mean()) para imputação
X = X.to_numpy()


X = np.nan_to_num(X, nan=0.0)
contagem_nan = np.sum(np.isnan(X))
print(f"Número total de valores NaN na matriz: {contagem_nan}")


Y = resultado_bruto['21490'].iloc[9:].to_numpy(dtype=float).reshape(-1, 1)

yhat = np.zeros(10)
for i in range(12, len(X)):
    #dividir X e Y em treino e teste
    X_train, X_test = X[:i], X[i-8:]
    Y_train, Y_test = Y[:i], Y[i-8:]

    modelo_hw.fit(X=X_train, y=Y_train)  # mesma defasagem usada antes
    print(i)
    yhat[i-12] = modelo_hw.predict(X=X_test, y=Y_test)[7]

# Flatten para 1D (necessário para fill_between)
y_true = Y_test.ravel()
y_pred = yhat.ravel()

plt.plot(y_true[8:], label='Real', color='blue')
plt.plot(y_pred, label='Previsto (H-W)', color='orange', linestyle='--')
#crie uma banda de 5% para cima e para baixo do resultado previsto
x_vals = np.arange(len(y_pred[8:]))
plt.fill_between(x_vals, y_pred[8:] * 0.95, y_pred[8:] * 1.05, color='orange', alpha=0.2)
plt.show()

rmse = np.sqrt(mean_squared_error(Y_test[8:], yhat[8:]))
rmse

Número total de valores NaN na matriz: 0
12
13
13
14
14
15
15


ValueError: Input X contains infinity or a value too large for dtype('float64').

In [ ]:
X[7:]

In [ ]:
X.shape

In [ ]:
# ...existing code...
# Preparar X uma única vez (todas empresas usam o mesmo)
X_df = exog_vars_estac_lags['2437'].iloc[:, 1:].apply(pd.to_numeric, errors='coerce').fillna(0.0)
X = X_df.to_numpy(dtype=float)

# Verificar NaN em X
contagem_nan = np.sum(np.isnan(X))
print(f"Número total de valores NaN na matriz X: {contagem_nan}\n")

# Definir xlag e modelo base
xlag = [[4] for _ in range(X.shape[1])]
ols = LeastSquares()

# Dicionários para armazenar resultados
hw_models = {}
hw_predictions = {}
hw_rmse = {}

# Loop por todas as empresas
for empresa in resultado_bruto.columns:
    print(f"\n{'='*60}")
    print(f"Empresa: {empresa}")
    print('='*60)
    
    # Preparar Y para esta empresa
    Y = resultado_bruto[empresa].iloc[9:].to_numpy(dtype=float).reshape(-1, 1)

    X_df = exog_vars_estac_lags['2437'].iloc[:, 1:].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    X = X_df.to_numpy(dtype=float)

    # Verificar NaN em X
    contagem_nan = np.sum(np.isnan(X))
    print(f"Número total de valores NaN na matriz X: {contagem_nan}\n")

    # Definir xlag e modelo base
    xlag = [[4] for _ in range(X.shape[1])]
    ols = LeastSquares()
    
    # Alinhar X e Y (pegar o menor tamanho)
    min_len = min(len(X), len(Y))
    X_aligned = X[-min_len:]
    Y_aligned = Y[-min_len:]
    
    # Dividir treino/teste
    X_train, X_test = X_aligned[:40], X_aligned[36:]
    Y_train, Y_test = Y_aligned[:40], Y_aligned[36:]
    
    print(f"Treino: X={X_train.shape}, Y={Y_train.shape}")
    print(f"Teste:  X={X_test.shape}, Y={Y_test.shape}")
    
    # Treinar modelo NARX
    modelo_hw = NARX(
        basis_function=Polynomial(degree=2),                      
        ylag=1,
        xlag=xlag,
        model_type="NARMAX",
        base_estimator=BayesianRidge()
    )
    
    # try:
    modelo_hw.fit(X=X_train, y=Y_train)
    # for i in range(0, 10):
    #     print(i)
    yhat = modelo_hw.predict(X=X_test, y=Y_test)
    modelo_hw.predict(X=X_test, y=Y_test)
    
    plt.plot(Y_test[8:], label='Real', color='blue')
    plt.plot(yhat[8:], label='Previsto (H-W)', color='orange', linestyle='--')
    plt.show()

    rmse = np.sqrt(mean_squared_error(Y_test[8:], yhat[8:]))
    
    # Armazenar resultados
    hw_models[empresa] = modelo_hw
    hw_predictions[empresa] = yhat
    hw_rmse[empresa] = rmse
    
    print(f"✓ RMSE: {rmse:.4f}")
        
    # except Exception as e:
    #     print(f"✗ Erro ao treinar modelo para {empresa}: {e}")
    #     hw_models[empresa] = None
    #     hw_predictions[empresa] = None
    #     hw_rmse[empresa] = np.nan

# Resumo final
print(f"\n{'='*60}")
print("RESUMO - RMSE por Empresa")
print('='*60)
rmse_df = pd.Series(hw_rmse, name='RMSE').sort_values()
display(rmse_df)
# ...existing code...

In [ ]:
len(Y_test)-1

In [ ]:
X_test[:8+i]

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler

In [ ]:
# ...existing code...
def estimar_hw_params(
    u,
    y,
    order_f,
    order_g,
    na,
    nb,
    maxiter=2000,
    n_restarts=5,
    random_state=None,
    verbose=False
):
    rng = np.random.default_rng(random_state)
    num_params = order_f + na + nb + order_g

    def _run_optim(x0):
        return minimize(
            objective_function,
            x0=x0,
            args=(u, y, order_f, order_g, na, nb),
            method="L-BFGS-B",
            options={"maxiter": maxiter, "eps": 1e-6, "ftol": 1e-9}
        )

    best_result = None

    # Primeira tentativa: vetor nulo (mais estável que aleatório grande)
    initial_guesses = [np.zeros(num_params)]
    # Tentativas extras: perturbações aleatórias pequenas
    for _ in range(max(0, n_restarts - 1)):
        initial_guesses.append(rng.normal(loc=0.0, scale=0.05, size=num_params))

    for attempt, x0 in enumerate(initial_guesses, start=1):
        try:
            result = _run_optim(x0)
        except Exception as exc:
            if verbose:
                print(f"[HW] Tentativa {attempt}: falha na otimização ({exc}).")
            continue

        if verbose and not result.success:
            print(f"[HW] Tentativa {attempt}: {result.message}")

        if best_result is None or result.fun < best_result.fun:
            best_result = result

        if result.success:
            break

    if best_result is None:
        raise RuntimeError("Otimização HW falhou em todas as tentativas.")

    if verbose and not best_result.success:
        print("[HW] Prosseguindo com melhor solução encontrada, embora sem convergência declarada.")

    return best_result.x, best_result
# ...existing code...
def walk_forward_hw_one_step(
    empresa,
    entrada_dict,
    saida_df,
    col_entrada=None,
    order_f=2,
    order_g=2,
    na=1,
    nb=2,
    corte_inicial=9,
    tamanho_treino_inicial=24,
    maxiter=2000,
    n_restarts=5,
    random_state=None,
    verbose=True
):
    u_df = entrada_dict[empresa].copy()
    y_series = saida_df[empresa].copy()
    if col_entrada is None:
        col_entrada = u_df.select_dtypes(include=[np.number]).columns[0]

    dados = (
        pd.concat([u_df[col_entrada], y_series], axis=1)
        .iloc[corte_inicial:]
        .dropna(how="any")
    )
    dados.columns = ["u_raw", "y_raw"]

    if dados.empty:
        raise ValueError("Nenhum dado disponível após corte e dropna.")

    atraso = max(na, nb)
    tamanho_minimo = max(tamanho_treino_inicial, atraso + 5)
    if tamanho_minimo >= len(dados):
        raise ValueError("Série curta demais para walk-forward com os parâmetros dados.")

    valores_reais = []
    previsoes = []
    indices_previsao = []
    params_hist = []
    params_prev = None

    if verbose:
        print(f"Walk-forward HW ({empresa}) - total pontos: {len(dados)}, previsões: {len(dados) - tamanho_minimo}")

    for i in range(tamanho_minimo, len(dados)):
        dados_treino = dados.iloc[:i]
        dados_teste = dados.iloc[i:i+1]

        scaler_u = StandardScaler()
        scaler_y = StandardScaler()

        u_train = scaler_u.fit_transform(dados_treino[["u_raw"]].to_numpy(dtype=float)).ravel()
        y_train = scaler_y.fit_transform(dados_treino[["y_raw"]].to_numpy(dtype=float)).ravel()

        try:
            params_est, opt_result = estimar_hw_params(
                u=u_train,
                y=y_train,
                order_f=order_f,
                order_g=order_g,
                na=na,
                nb=nb,
                maxiter=maxiter,
                n_restarts=n_restarts,
                random_state=random_state,
                verbose=verbose
            )
            params_prev = params_est
        except RuntimeError as exc:
            if params_prev is None:
                if verbose:
                    print(f"[HW] Passo {i - tamanho_minimo + 1}: falha sem parâmetros prévios ({exc}). Inserindo NaN.")
                previsoes.append(np.nan)
                valores_reais.append(float(dados_teste["y_raw"].iloc[0]))
                indices_previsao.append(dados_teste.index[0])
                params_hist.append(None)
                continue
            if verbose:
                print(f"[HW] Passo {i - tamanho_minimo + 1}: reutilizando parâmetros anteriores devido a {exc}.")
            params_est = params_prev

        u_full = scaler_u.transform(dados.iloc[:i+1][["u_raw"]].to_numpy(dtype=float)).ravel()
        y_hat_norm = simulate_hw_model(params_est, u_full, order_f, order_g, na, nb)
        y_hat_last = scaler_y.inverse_transform(y_hat_norm[-1:].reshape(-1, 1)).ravel()[0]

        y_real_last = float(dados_teste["y_raw"].iloc[0])

        previsoes.append(y_hat_last)
        valores_reais.append(y_real_last)
        indices_previsao.append(dados_teste.index[0])
        params_hist.append(params_est)

        if verbose and ((i - tamanho_minimo + 1) % 5 == 0 or i == len(dados) - 1):
            rmse_parcial = np.sqrt(mean_squared_error(valores_reais, np.nan_to_num(previsoes, nan=np.nanmean(previsoes))))
            print(f"Passo {i - tamanho_minimo + 1}/{len(dados) - tamanho_minimo} | RMSE parcial: {rmse_parcial:.4f}")

    df_resultados = pd.DataFrame(
        {"y_real": valores_reais, "y_hat": previsoes},
        index=pd.Index(indices_previsao, name="data")
    )
    rmse_final = float(np.sqrt(mean_squared_error(valores_reais, np.nan_to_num(previsoes, nan=np.nanmean(previsoes)))))

    resumo = {
        "empresa": empresa,
        "col_entrada": col_entrada,
        "ordens": {"order_f": order_f, "order_g": order_g, "na": na, "nb": nb},
        "params_hist": params_hist,
        "rmse": rmse_final,
        "resultado": df_resultados
    }

    if verbose:
        print(f"RMSE final (one-step, walk-forward): {rmse_final:.4f}")

    return resumo

In [ ]:
def estimar_hw_params(
    u,
    y,
    order_f,
    order_g,
    na,
    nb,
    maxiter=2000,
    n_restarts=5,
    random_state=None,
    verbose=False
):
    rng = np.random.default_rng(random_state)
    num_params = order_f + na + nb + order_g

    initial_guesses = [np.zeros(num_params)]
    for _ in range(max(0, n_restarts - 1)):
        initial_guesses.append(rng.normal(loc=0.0, scale=0.05, size=num_params))

    best_result = None
    for attempt, x0 in enumerate(initial_guesses, start=1):
        try:
            result = minimize(
                objective_function,
                x0=x0,
                args=(u, y, order_f, order_g, na, nb),
                method="L-BFGS-B",
                options={"maxiter": maxiter, "eps": 1e-6, "ftol": 1e-9}
            )
        except Exception as exc:
            if verbose:
                print(f"[HW] Tentativa {attempt}: otimização falhou ({exc}).")
            continue

        if verbose and not result.success:
            print(f"[HW] Tentativa {attempt}: {result.message}")

        if best_result is None or result.fun < best_result.fun:
            best_result = result
        if result.success:
            break

    if best_result is None:
        raise RuntimeError("Otimização HW falhou em todas as tentativas.")

    if verbose and not best_result.success:
        print("[HW] Prosseguindo com melhor solução encontrada, embora sem convergência declarada.")

    return best_result.x, best_result

def walk_forward_hw_one_step(
    empresa,
    entrada_dict,
    saida_df,
    col_entrada=None,
    order_f=2,
    order_g=2,
    na=1,
    nb=2,
    corte_inicial=9,
    tamanho_treino_inicial=24,
    maxiter=2000,
    n_restarts=5,
    random_state=None,
    verbose=True
):
    # 1º BLOCO NÃO LINEAR (pré-processamento) ---------------------------------
    # aqui aplicamos a transformação não linear na entrada (ex.: polinômios)
    def nonlinear_in(u_raw):
        return u_raw  # substitua para testar outra não linearidade (ex.: np.tanh)

    # BLOCO LINEAR DINÂMICO ----------------------------------------------------
    # parâmetros na/nb governam a parte ARX (dinâmica linear) simulada por simulate_hw_model

    # 2º BLOCO NÃO LINEAR (pós-processamento) ----------------------------------
    # transformação final sobre a saída linear (ex.: função identidade)
    def nonlinear_out(y_lin):
        return y_lin  # altere para testar outra não linearidade (ex.: np.exp)

    ################ APENAS TRATAMENTOS INICIAIS ####################################
    u_df = entrada_dict[empresa].copy()
    y_series = saida_df[empresa].copy()
    if col_entrada is None:
        col_entrada = u_df.select_dtypes(include=[np.number]).columns[0]

    dados = (
        pd.concat([u_df[col_entrada], y_series], axis=1)
        .iloc[corte_inicial:]
        .dropna(how="any")
    )
    dados.columns = ["u_raw", "y_raw"]

    if dados.empty:
        raise ValueError("Nenhum dado disponível após corte e dropna.")

    atraso = max(na, nb)
    tamanho_minimo = max(tamanho_treino_inicial, atraso + 5)
    if tamanho_minimo >= len(dados):
        raise ValueError("Série curta demais para walk-forward com os parâmetros dados.")

    valores_reais, previsoes, indices_previsao = [], [], []
    params_hist = []
    params_prev = None

    if verbose:
        print(f"Walk-forward HW ({empresa}) - total pontos: {len(dados)}, previsões: {len(dados) - tamanho_minimo}")

    ################ APENAS TRATAMENTOS INICIAIS - FIM ####################################

    for i in range(tamanho_minimo, len(dados)):
        dados_treino = dados.iloc[:i]
        dados_teste = dados.iloc[i:i+1]

        scaler_u = StandardScaler()
        scaler_y = StandardScaler()

        u_train = scaler_u.fit_transform(nonlinear_in(dados_treino[["u_raw"]]).to_numpy(dtype=float)).ravel()
        y_train = scaler_y.fit_transform(dados_treino[["y_raw"]].to_numpy(dtype=float)).ravel()

        try:
            params_est, _ = estimar_hw_params(
                u=u_train,
                y=y_train,
                order_f=order_f,
                order_g=order_g,
                na=na,
                nb=nb,
                maxiter=maxiter,
                n_restarts=n_restarts,
                random_state=random_state,
                verbose=verbose
            )
            params_prev = params_est
        except RuntimeError as exc:
            if params_prev is None:
                if verbose:
                    print(f"[HW] Passo {i - tamanho_minimo + 1}: falha sem parâmetros prévios ({exc}). Inserindo NaN.")
                previsoes.append(np.nan)
                valores_reais.append(float(dados_teste["y_raw"].iloc[0]))
                indices_previsao.append(dados_teste.index[0])
                params_hist.append(None)
                continue
            if verbose:
                print(f"[HW] Passo {i - tamanho_minimo + 1}: reutilizando parâmetros anteriores ({exc}).")
            params_est = params_prev

        u_full = scaler_u.transform(nonlinear_in(dados.iloc[:i+1][["u_raw"]]).to_numpy(dtype=float)).ravel()
        y_hat_lin = simulate_hw_model(params_est, u_full, order_f, order_g, na, nb)
        y_hat_last = scaler_y.inverse_transform(nonlinear_out(y_hat_lin[-1:]).reshape(-1, 1)).ravel()[0]

        y_real_last = float(dados_teste["y_raw"].iloc[0])

        previsoes.append(y_hat_last)
        valores_reais.append(y_real_last)
        indices_previsao.append(dados_teste.index[0])
        params_hist.append(params_est)

        if verbose and ((i - tamanho_minimo + 1) % 5 == 0 or i == len(dados) - 1):
            rmse_parcial = np.sqrt(mean_squared_error(valores_reais, np.nan_to_num(previsoes, nan=np.nanmean(previsoes))))
            print(f"Passo {i - tamanho_minimo + 1}/{len(dados) - tamanho_minimo} | RMSE parcial: {rmse_parcial:.4f}")

    df_resultados = pd.DataFrame(
        {"y_real": valores_reais, "y_hat": previsoes},
        index=pd.Index(indices_previsao, name="data")
    )
    rmse_final = float(np.sqrt(mean_squared_error(valores_reais, np.nan_to_num(previsoes, nan=np.nanmean(previsoes)))))

    resumo = {
        "empresa": empresa,
        "col_entrada": col_entrada,
        "ordens": {"order_f": order_f, "order_g": order_g, "na": na, "nb": nb},
        "params_hist": params_hist,
        "rmse": rmse_final,
        "resultado": df_resultados
    }

    if verbose:
        print(f"RMSE final (one-step, walk-forward): {rmse_final:.4f}")

    return resumo

# parâmetros principais:
# order_f/order_g -> grau dos blocos não lineares (f e g)
# na -> número de lags da saída (parte AR)
# nb -> número de lags da entrada
# corte_inicial/tamanho_treino_inicial -> controles de janela temporal
# maxiter/n_restarts -> esforço de otimização

In [ ]:
res_wf = walk_forward_hw_one_step(
    empresa=empresa_alvo,
    entrada_dict=input_receita,
    saida_df=resultado_bruto,
    col_entrada=None,
    order_f=2,
    order_g=2,
    na=1,
    nb=2,
    corte_inicial=9,
    tamanho_treino_inicial=12,
    maxiter=1000,
    n_restarts=6,
    random_state=42,
    verbose=True
)